# 14. Imputed columns and the decimal lattice, on top of target encoding

Two feature blocks, two flags, one variable per run. The encoder and the fold
scheme are unchanged from `13`, so every run here is comparable to ledger row 17
(`lgbm_bag08_seed42_te`, CV 0.966782) and to each other.

**Imputed columns, added alongside the NaNs and never replacing them.** LightGBM
learns a default split direction per node for a missing value, which is strictly
more expressive than one imputed point estimate. Replacing the column throws that
away; adding a second column keeps both the missingness structure and a value the
model can use in arithmetic. The median is computed inside the fold loop, on
training rows only.

**The decimal lattice.** The first decimal digit of the continuous columns carries
a target-rate swing that has nothing to do with anyone's phone use. It is a
fingerprint of how the generator wrote the numbers. Target encoding cannot see it,
because TE estimates each exact value independently and has no way to express
"everything ending in .2 shares something" — that statement pools across integer
parts and TE's levels do not. A different channel, not a different view.

Both ideas are from the public notebook by tomasa2, which measured +0.0012 and
+0.0001. Our own implementation, our own folds, our own numbers.


In [ ]:
SMOKE = True

# One variable per run. Row 17 is both flags False.
USE_IMPUTED = True
USE_LATTICE = False

SEED = 42
N_INNER = 5
SMOOTH = 10.0
BASELINE_CV = 0.966782        # ledger row 17, target encoding only
EXPECTED_FOLD_SHA = "ec282b0968059676"

PARAMS = dict(
    objective="binary", metric="auc", learning_rate=0.05, n_estimators=2000,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=SEED, n_jobs=-1, verbose=-1,
    deterministic=True, force_row_wise=True,
)
print(f"SMOKE={SMOKE}  USE_IMPUTED={USE_IMPUTED}  USE_LATTICE={USE_LATTICE}")


In [ ]:
import hashlib
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold


def locate(name):
    kag = Path("/kaggle/input")
    if kag.exists():
        hits = sorted(kag.rglob(name))
        if hits:
            return hits[0]
    for b in [Path.cwd(), *Path.cwd().parents]:
        p = b / "data" / "raw" / name
        if p.exists():
            return p
    raise FileNotFoundError(name)


train = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
TARGET = "addicted_label"
CAT = ["gender", "stress_level", "academic_work_impact"]
RAW = [c for c in train.columns if c not in ("id", TARGET)]
NUM = [c for c in RAW if c not in CAT]
# Columns that actually carry decimals. age, notifications_per_day and
# app_opens_per_day are integer-valued, so a fractional part would be a constant.
FLOATY = ["daily_screen_time_hours", "social_media_hours", "gaming_hours",
          "work_study_hours", "sleep_hours", "weekend_screen_time"]

if SMOKE:
    train = train.sample(20000, random_state=0).reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    PARAMS["n_estimators"] = 200

y = train[TARGET].to_numpy()
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i
sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print(f"rows {len(train)}   fold sha {sha}")
print("SMOKE: sha expected to differ" if SMOKE else
      ("fold alignment: VERIFIED" if ALIGNED else "fold alignment: MISMATCH"))


In [ ]:
def add_lattice(df):
    """Fractional part and first decimal digit. Deterministic, no target, no fit."""
    out = df.copy()
    for c in FLOATY:
        v = df[c].to_numpy(dtype=float)
        frac = v - np.floor(v)
        out[f"frac_{c}"] = frac
        # round before casting: 8.3 is 8.2999... in binary and would give 2.
        d1 = np.where(np.isnan(v), -1, np.round(frac * 10) % 10)
        out[f"d1_{c}"] = d1
    return out


X = train[RAW].copy()
X_test = test[RAW].copy()
if USE_LATTICE:
    X = add_lattice(X)
    X_test = add_lattice(X_test)
    # The digit is a label, not a magnitude, so it is target-encoded like any
    # other level rather than left as a number the tree has to threshold.
    TE_COLS = RAW + [f"d1_{c}" for c in FLOATY]
else:
    TE_COLS = list(RAW)

for c in CAT:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")

print(f"{X.shape[1]} columns before encoding, {len(TE_COLS)} of them encoded")


In [ ]:
def _stats(levels, yy, prior):
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    return ((g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH),
            g["count"] / len(df))


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt):
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(tr))
    for c in TE_COLS:
        lv = Xf[c].to_numpy()
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean, freq, prior)
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf
    return pd.DataFrame(e_tr), pd.DataFrame(e_va), pd.DataFrame(e_te)


def build(Xf, yy, tr, va, Xt):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = pd.concat([Xt.reset_index(drop=True), d_te], axis=1)

    if USE_IMPUTED:
        # Median from TRAINING rows of this fold only. Added as extra columns;
        # the originals keep their NaNs so the model still sees missingness.
        med = Xf.iloc[tr][NUM].median()
        for c in NUM:
            Xtr[f"imp_{c}"] = Xf.iloc[tr][c].fillna(med[c]).to_numpy()
            Xva[f"imp_{c}"] = Xf.iloc[va][c].fillna(med[c]).to_numpy()
            Xte[f"imp_{c}"] = Xt[c].fillna(med[c]).to_numpy()
    return Xtr, Xva, Xte


In [ ]:
# Leak checks, same three as notebook 13. Printed, not asserted.
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va, X_test)

y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va, X_test)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in TE_COLS)

_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va, X_test)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in TE_COLS)
shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va, X_test)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in TE_COLS)

print(f"1. validation rows, own target flipped : {leak1:.3e}   (must be 0)")
print(f"2. training rows, own target flipped   : {leak2:.3e}   "
      f"(prior moved {shift:.3e}; these track each other)")
print(f"3. training targets flipped, val moves : {live:.3e}   (must be large)")
CLEAN = leak1 == 0 and leak2 < 10 * max(shift, 1e-9) and live > 0.1
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")


In [ ]:
oof = np.zeros(len(train))
test_pred = np.zeros(len(test))
scores = []
t0 = time.time()

for f in range(5):
    tr = np.where(folds != f)[0]
    va = np.where(folds == f)[0]
    Xtr, Xva, Xte = build(X, y, tr, va, X_test)
    if f == 0:
        print(f"{Xtr.shape[1]} features reach the model")
    m = lgb.LGBMClassifier(**PARAMS).fit(Xtr, y[tr])
    p = m.predict_proba(Xva)[:, 1]
    oof[va] = p
    test_pred += m.predict_proba(Xte)[:, 1] / 5
    scores.append(roc_auc_score(y[va], p))
    print(f"  fold {f}: {scores[-1]:.6f}   [{(time.time() - t0) / 60:.1f} min]")

cv, sd = float(np.mean(scores)), float(np.std(scores))
print()
print(f"CV {cv:.6f} +/- {sd:.6f}")
print(f"row 17 (TE only) {BASELINE_CV:.6f}   diff {cv - BASELINE_CV:+.6f}")
print(f"relative {(cv - BASELINE_CV) / BASELINE_CV:+.4%}  (CLAUDE.md flags >2%)")


In [ ]:
tag = f"te{'_imp' if USE_IMPUTED else ''}{'_lat' if USE_LATTICE else ''}"
prefix = "SMOKE_" if SMOKE else ""
out = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

np.save(out / f"{prefix}{tag}_oof.npy", oof)
np.save(out / f"{prefix}{tag}_test.npy", test_pred)
pd.DataFrame({"id": test["id"], TARGET: test_pred}).to_csv(
    out / f"{prefix}submission.csv", index=False)

print(f"wrote {prefix}{tag}_oof.npy and {prefix}submission.csv")
print()
print(f"ledger line: lgbm_bag08_seed42_{tag}  cv {cv:.6f} +/- {sd:.6f}  "
      f"leaks {'PASS' if CLEAN else 'FAIL'}  aligned {ALIGNED}")
